# Math Reasoning Generator — OpenAI o3-mini (Preview + Multi-Batch)
- **Cell 1**: Install
- **Cell 2**: Config
- **Cell 3**: Helpers
- **Cell 4**: Preview first 5 questions in real-time
- **Cell 5**: Split remaining questions into batches of 100 and submit all
- **Cell 6**: Poll all batches — as each finishes, results are written to the final JSONL immediately


In [ ]:
# Cell 1 — Install
!pip install openai -q

In [ ]:
# Cell 2 — Config
import os

os.environ['OPENAI_API_KEY'] = ''

INPUT_PATH        = 'data/public.jsonl'
OUTPUT_PATH       = 'data/public_with_ai_reasoning.jsonl'  # final merged output
PREVIEW_PATH      = 'data/preview_results.jsonl'        # first 5 real-time results
BATCH_DIR         = 'batches'                      # folder for batch input/output files
BATCH_IDS_PATH    = 'batch_ids.json'               # tracks all submitted batch IDs

MODEL             = 'o3-mini'
MAX_TOKENS        = 12000
REASONING_EFFORT  = 'high'   # 'low', 'medium', 'high'
PREVIEW_N         = 5          # questions to run in real-time first
BATCH_SIZE        = 100        # questions per batch


In [ ]:
# Cell 3 — Imports & helpers
import json, re, time, os
from pathlib import Path
from openai import OpenAI, RateLimitError

client = OpenAI()
os.makedirs(BATCH_DIR, exist_ok=True)

# ── Prompts ────────────────────────────────────────────────────────────
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician reasoning. Think concisely. Step by step.\n\n "
    "MANDATORY RULE:\n"
    "- Show ALL reasoning inside <think>...</think> tags."
    "- The FINAL token MUST be: \\boxed{final_answer}\n "
    "- If multiple sub-answers are required, put them in one box separated by commas, e.g. \\boxed{3, 7}.\n "
    "- If this is not the final token, the response is invalid.\n\n"
    "FORMAT:\n"
    "<think>\n"
    "[your reasoning here]\n"
    "</think>\n"
    "\\boxed{final_answer}\n"
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician reasoning. Think concisely. Step by step.\n\n"
    "MANDATORY RULE:\n"
    "- Show ALL reasoning inside <think>...</think> tags."
    "- The FINAL token MUST be: \\boxed{X} where X is the answer choice given the options A-Z.\n "
    "- If this is not the final token, the response is invalid.\n\n"
    "FORMAT:\n"
    "<think>\n"
    "[your reasoning here]\n"
    "</think>\n"
    "\\boxed{X}\n"
)

# ── Helpers ────────────────────────────────────────────────────────────
def get_system_prompt(item):
    return SYSTEM_PROMPT_MCQ if item.get('options') else SYSTEM_PROMPT_MATH

def build_user_prompt(item):
    msg = f"Question:\n{item['question']}"
    if item.get('options'):
        labeled = '\n'.join(f"{chr(65+i)}. {o}" for i, o in enumerate(item['options']))
        msg += f"\n\nOptions:\n{labeled}"
    return msg

def extract_think(text):
    m = re.search(r'<think>([\s\S]*?)</think>', text)
    if m:
        return m.group(1).strip()
    bi = text.find('\\boxed')
    if bi > 0:
        return text[:bi].strip()
    return text.strip()

def call_api_single(item, retries=3):
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                max_completion_tokens=MAX_TOKENS,
                reasoning_effort=REASONING_EFFORT,
                messages=[
                    {'role': 'system', 'content': get_system_prompt(item)},
                    {'role': 'user',   'content': build_user_prompt(item)}
                ]
            )
            text = response.choices[0].message.content or ''
            if '<think>' in text and '</think>' in text and '\\boxed' in text:
                return extract_think(text), text  # (reasoning, raw)
            print(f"  Bad format on id={item['id']} (attempt {attempt+1}), retrying...")
            print(f"  Response was: {text[:200]!r}")
        except RateLimitError:
            wait = 15 * (attempt + 1)
            print(f"  Rate limited on id={item['id']}, waiting {wait}s...")
            time.sleep(wait)
        except Exception as e:
            wait = 5 * (attempt + 1)
            print(f"  Error on id={item['id']}: {e}, waiting {wait}s...")
            time.sleep(wait)
    return None, None

def load_done_ids(path):
    done = set()
    if Path(path).exists():
        with open(path) as f:
            for line in f:
                try: done.add(json.loads(line.strip())['id'])
                except: pass
    return done

def append_to_output(rows):
    """Append a list of result dicts to the final output file."""
    with open(OUTPUT_PATH, 'a', buffering=1) as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')

def make_row(item, reasoning, raw):
    return {
        'id':             item['id'],
        'question':       item['question'],
        'options':        item.get('options', None),
        'answer':         item['answer'],
        'reasoning':      reasoning,
        'reasoning_raw':  raw,
    }

# Load dataset
items = [json.loads(l) for l in open(INPUT_PATH) if l.strip()]
id_to_item = {str(x['id']): x for x in items}
print(f'Loaded {len(items)} questions.')


In [ ]:
# Cell 4 — Preview: run first N questions in real-time
# Lets you verify format before committing to the full batch.

preview_items = items[:PREVIEW_N]
done_ids = load_done_ids(PREVIEW_PATH)
todo_preview = [x for x in preview_items if x['id'] not in done_ids]
print(f'Running {len(todo_preview)} preview questions...\n')

preview_rows = []
with open(PREVIEW_PATH, 'a', buffering=1) as f:
    for i, item in enumerate(todo_preview):
        print(f'[{i+1}/{len(todo_preview)}] id={item["id"]} generating...')
        reasoning, raw = call_api_single(item)
        if reasoning:
            row = make_row(item, reasoning, raw)
            f.write(json.dumps(row, ensure_ascii=False) + '\n')
            preview_rows.append(row)
            print(f'  ✓ ({len(reasoning)} chars)')
            print(f'  --- REASONING PREVIEW ---')
            print(reasoning[:500] + ('...' if len(reasoning) > 500 else ''))
            print()
        else:
            print(f'  ✗ FAILED')

# Write preview results to final output immediately
if preview_rows:
    append_to_output(preview_rows)
    print(f'\n{len(preview_rows)} preview results written to {OUTPUT_PATH}.')
print('\nIf format looks good, run Cell 5 to submit batches.')


In [ ]:
# Cell 5 — Split remaining questions into batches of BATCH_SIZE and submit all

done_ids = load_done_ids(OUTPUT_PATH)
batch_items = [x for x in items if x['id'] not in done_ids]
chunks = [batch_items[i:i+BATCH_SIZE] for i in range(0, len(batch_items), BATCH_SIZE)]
print(f'{len(batch_items)} questions → {len(chunks)} batches of up to {BATCH_SIZE}')

# Load existing batch IDs (resume support: skip already-submitted batches)
batch_registry = {}
if Path(BATCH_IDS_PATH).exists():
    batch_registry = json.load(open(BATCH_IDS_PATH))
    print(f'Loaded {len(batch_registry)} existing batch submissions.')

for idx, chunk in enumerate(chunks):
    chunk_key = f'batch_{idx:03d}'
    if chunk_key in batch_registry:
        print(f'  [{idx+1}/{len(chunks)}] {chunk_key} already submitted ({batch_registry[chunk_key]["batch_id"]}) — skipping')
        continue

    # Write batch input file
    input_path = f'{BATCH_DIR}/{chunk_key}_input.jsonl'
    with open(input_path, 'w') as f:
        for item in chunk:
            request = {
                'custom_id': str(item['id']),
                'method': 'POST',
                'url': '/v1/chat/completions',
                'body': {
                    'model': MODEL,
                    'max_completion_tokens': MAX_TOKENS,
                    'reasoning_effort': REASONING_EFFORT,
                    'messages': [
                        {'role': 'system', 'content': get_system_prompt(item)},
                        {'role': 'user',   'content': build_user_prompt(item)}
                    ]
                }
            }
            f.write(json.dumps(request) + '\n')

    # Upload and submit
    with open(input_path, 'rb') as f:
        uploaded = client.files.create(file=f, purpose='batch')
    batch = client.batches.create(
        input_file_id=uploaded.id,
        endpoint='/v1/chat/completions',
        completion_window='24h'
    )
    batch_registry[chunk_key] = {
        'batch_id':   batch.id,
        'status':     batch.status,
        'n_items':    len(chunk),
        'downloaded': False
    }
    json.dump(batch_registry, open(BATCH_IDS_PATH, 'w'), indent=2)
    print(f'  [{idx+1}/{len(chunks)}] {chunk_key} submitted → {batch.id} ({len(chunk)} questions)')
    time.sleep(1)  # avoid hammering the API

print(f'\nAll batches submitted. Run Cell 6 to poll for results.')


In [ ]:
# Cell 6 — Poll all batches; write results to final JSONL as each batch completes
# Re-run this cell at any time to check progress and pick up finished batches.

if not Path(BATCH_IDS_PATH).exists():
    print('No batch_ids.json found. Run Cell 5 first.')
else:
    batch_registry = json.load(open(BATCH_IDS_PATH))
    total_batches  = len(batch_registry)

    while True:
        batch_registry = json.load(open(BATCH_IDS_PATH))  # reload in case of resume
        pending = [k for k, v in batch_registry.items() if not v['downloaded']]

        if not pending:
            print('All batches downloaded and merged into output file!')
            break

        print(f'\n--- Polling {len(pending)} pending batches ---')
        all_done = True

        for key in sorted(pending):
            entry    = batch_registry[key]
            batch_id = entry['batch_id']
            batch    = client.batches.retrieve(batch_id)
            counts   = batch.request_counts
            status   = batch.status
            print(f'  {key} [{batch_id}]: {status} | '
                  f'completed={counts.completed}/{counts.total} failed={counts.failed}')

            if status == 'completed':
                # Download output
                output_path = f'{BATCH_DIR}/{key}_output.jsonl'
                content = client.files.content(batch.output_file_id)
                with open(output_path, 'wb') as f:
                    f.write(content.read())

                # Parse and write results immediately to final output
                rows = []
                errors = 0
                for line in open(output_path):
                    try:
                        resp = json.loads(line.strip())
                        cid  = resp['custom_id']
                        item = id_to_item.get(cid)
                        if not item: continue
                        text = (resp.get('response', {})
                                    .get('body', {})
                                    .get('choices', [{}])[0]
                                    .get('message', {})
                                    .get('content', ''))
                        if '<think>' in text and '</think>' in text and '\\boxed' in text:
                            reasoning = extract_think(text)
                            rows.append(make_row(item, reasoning, text))
                        else:
                            errors += 1
                            print(f'    Bad format for id={cid}: {text[:100]!r}')
                    except Exception as e:
                        errors += 1
                        print(f'    Parse error: {e}')

                append_to_output(rows)
                batch_registry[key]['downloaded'] = True
                batch_registry[key]['status'] = 'completed'
                json.dump(batch_registry, open(BATCH_IDS_PATH, 'w'), indent=2)

                done_so_far = sum(1 for v in batch_registry.values() if v['downloaded'])
                print(f'    ✓ Written {len(rows)} rows ({errors} errors) → '
                      f'{OUTPUT_PATH} [{done_so_far}/{total_batches} batches done]')

            elif status in ('failed', 'expired', 'cancelled'):
                print(f'    ✗ Batch {status} — skipping')
                batch_registry[key]['downloaded'] = True
                batch_registry[key]['status'] = status
                json.dump(batch_registry, open(BATCH_IDS_PATH, 'w'), indent=2)
            else:
                all_done = False  # still in_progress or validating

        pending_after = [k for k, v in batch_registry.items() if not v['downloaded']]
        if not pending_after:
            print('\nAll batches complete!')
            break
        if all_done:
            break
        print(f'\n{len(pending_after)} batches still running. Waiting 60s...')
        time.sleep(60)

    # Final count
    total_written = sum(1 for _ in open(OUTPUT_PATH)) if Path(OUTPUT_PATH).exists() else 0
    print(f'\nFinal output: {total_written} rows in {OUTPUT_PATH}')

In [ ]:
import json

new_row = {
    "id": 0,
    "question": "Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]",
    "options": None,
    "answer": "325*(1+325)",
    "reasoning": "The first 325 positive even numbers can be written as 2, 4, 6, …, 2×325. Their sum S is:   S = 2 + 4 + 6 + … + (2×325) = 2(1 + 2 + 3 + … + 325). Using the formula for the sum of the first N natural numbers, where N = 325:   1 + 2 + 3 + … + 325 = (325×326)/2. Substitute this back into the sum:   S = 2 × [(325×326)/2] = 325×326. Now, calculate 325×326:   325×326 = 325×(300 + 26) = 325×300 + 325×26 = 97500 + 8450 = 105950.",
    "reasoning_raw": "<think> The first 325 positive even numbers can be written as 2, 4, 6, …, 2×325. Their sum S is:   S = 2 + 4 + 6 + … + (2×325) = 2(1 + 2 + 3 + … + 325). Using the formula for the sum of the first N natural numbers, where N = 325:   1 + 2 + 3 + … + 325 = (325×326)/2. Substitute this back into the sum:   S = 2 × [(325×326)/2] = 325×326. Now, calculate 325×326:   325×326 = 325×(300 + 26) = 325×300 + 325×26 = 97500 + 8450 = 105950. </think> \\boxed{105950}"
}

with open("data/public_with_ai_reasoning.jsonl", "a") as f:
    f.write(json.dumps(new_row) + "\n")

In [ ]:
import pandas as pd, re, sys, json
from tqdm import tqdm

# ── 1. Convert JSONL to CSV ──────────────────────────────────────────────────
with open("data/public_with_ai_reasoning.jsonl", "r") as f:
    raw_data = [json.loads(line) for line in f]

# CSV has just question + reasoning_raw (the model's response)
csv_df = pd.DataFrame([{
    "id":           row["id"],
    "question":     row["question"],
    "revised_response": row["reasoning_raw"]   # renamed to match reference script
} for row in raw_data])

csv_df.to_csv("data/responses_checkpoint.csv", index=False)
print(f"Converted {len(csv_df)} rows to data/responses_checkpoint.csv")

# ── 2. Score ─────────────────────────────────────────────────────────────────
df   = pd.read_csv("data/responses_checkpoint.csv")
data = raw_data  # ground truth lookup

print(f"Loaded {len(df)} responses from checkpoint")

def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", str(text).upper())
    return matches[-1] if matches else ""

def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()

sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

data_lookup = {item["id"]: item for item in data}

results = []
for _, row in tqdm(df.iterrows(), total=len(df), desc="Scoring"):
    item = data_lookup.get(row["id"])
    if item is None:
        continue
    response = row["revised_response"]
    is_mcq   = bool(item.get("options"))
    gold     = item["answer"]

    if is_mcq:
        correct = score_mcq(str(response), str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       row["id"],
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

results_df = pd.DataFrame(results)
total    = len(results_df)
correct  = results_df["correct"].sum()
mcq_acc  = results_df[results_df["is_mcq"]]["correct"].mean()
free_acc = results_df[~results_df["is_mcq"]]["correct"].mean()

print(f"\nScoring complete: {total} results")
print(f"Overall accuracy:    {correct/total:.4f} ({int(correct)}/{total})")
print(f"MCQ accuracy:        {mcq_acc:.4f}")
print(f"Free-form accuracy:  {free_acc:.4f}")

results_df.to_csv("data/scoring_results.csv", index=False)
print("Saved to data/scoring_results.csv")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# SCRIPT 2: split_correct_incorrect.py
# Splits scored rows into two JSONLs:
#   data/correct.jsonl   — rows the model got right (for LoRA training)
#   data/incorrect.jsonl — rows the model got wrong (for recovery)
# reasoning_raw is stripped from both since we'll regenerate it
# ═══════════════════════════════════════════════════════════════════
import json, pandas as pd

# Load scoring results from script 1
results_df = pd.read_csv("data/scoring_results.csv")

# Load original data to get full row content
with open("data/public_with_ai_reasoning.jsonl", "r") as f:
    raw_data = {json.loads(line)["id"]: json.loads(line) for line in f}

# Re-open since we consumed the iterator above
with open("data/public_with_ai_reasoning.jsonl", "r") as f:
    raw_data = {}
    for line in f:
        row = json.loads(line)
        raw_data[row["id"]] = row

correct_ids   = set(results_df[results_df["correct"] == True]["id"])
incorrect_ids = set(results_df[results_df["correct"] == False]["id"])

with open("data/correct.jsonl", "w") as fc, \
     open("data/incorrect.jsonl", "w") as fi:
    for id_, row in raw_data.items():
        # keep reasoning_raw in correct rows — these are good training examples
        # strip it from incorrect rows — we'll regenerate better reasoning
        if id_ in correct_ids:
            fc.write(json.dumps(row) + "\n")
        elif id_ in incorrect_ids:
            row_copy = {k: v for k, v in row.items() if k != "reasoning_raw"}
            fi.write(json.dumps(row_copy) + "\n")

print(f"Correct rows:   {len(correct_ids)} → data/correct.jsonl")
print(f"Incorrect rows: {len(incorrect_ids)} → data/incorrect.jsonl")

In [ ]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician reasoning. Think concisely. Step by step.\n\n "
    "MANDATORY RULE:\n"
    "- Show ALL reasoning inside <think>...</think> tags."
    "- The FINAL token MUST be: \\boxed{final_answer}\n "
    "- If multiple sub-answers are required, put them in one box separated by commas, e.g. \\boxed{3, 7}.\n "
    "- If this is not the final token, the response is invalid.\n\n"
    "FORMAT:\n"
    "<think>\n"
    "[your reasoning here]\n"
    "</think>\n"
    "\\boxed{final_answer}\n"
)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# SCRIPT 3a: recover_incorrect_batch_submit.py
# Submits backward reasoning requests as multiple smaller OpenAI
# batch jobs (50% discount). Smaller batches run faster.
# Run this first, then run 3b to collect results.
# ═══════════════════════════════════════════════════════════════════
import json, os
from openai import OpenAI

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician reasoning. Think concisely. Step by step.\n\n "
    "MANDATORY RULE:\n"
    "- Show ALL reasoning inside <think>...</think> tags."
    "- The FINAL token MUST be: \\boxed{final_answer}\n "
    "- If multiple sub-answers are required, put them in one box separated by commas, e.g. \\boxed{3, 7}.\n "
    "- If this is not the final token, the response is invalid.\n\n"
    "FORMAT:\n"
    "<think>\n"
    "[your reasoning here]\n"
    "</think>\n"
    "\\boxed{final_answer}\n"
)

BATCH_SIZE = 50

# ── Load only free-form incorrect rows ──────────────────────────
with open("data/incorrect.jsonl", "r") as f:
    incorrect = [json.loads(line) for line in f]

free_form_incorrect = [r for r in incorrect if not r.get("options")]
print(f"Free-form incorrect rows: {len(free_form_incorrect)}")

# ── Split into chunks ────────────────────────────────────────────
chunks = [free_form_incorrect[i:i+BATCH_SIZE]
          for i in range(0, len(free_form_incorrect), BATCH_SIZE)]
print(f"Splitting into {len(chunks)} batches of ~{BATCH_SIZE}")

# ── Submit each chunk as a separate batch ────────────────────────
batch_ids = []

for idx, chunk in enumerate(chunks):
    batch_requests = []
    for row in chunk:
        question = row["question"]
        gold     = str(row["answer"]).strip()

        prompt = (
            f"Solve this math problem carefully.\n\n{question}\n\n"
            f"The correct answer is {gold}. "
            f"Show the complete step-by-step reasoning that arrives at this answer."
        )

        batch_requests.append({
            "custom_id": str(row["id"]),
            "method": "POST",
            "url": "/v1/chat/completions",
            "body": {
                "model": "o4-mini",
                "reasoning_effort": "high",
                "max_completion_tokens": 12000,
                "messages": [
                    {"role": "system", "content": SYSTEM_PROMPT_MATH},
                    {"role": "user",   "content": prompt}
                ]
            }
        })

    # write chunk to temp file
    batch_input_path = f"data/recover_batch_input_{idx}.jsonl"
    with open(batch_input_path, "w") as f:
        for req in batch_requests:
            f.write(json.dumps(req) + "\n")

    # upload and submit
    with open(batch_input_path, "rb") as f:
        uploaded = client.files.create(file=f, purpose="batch")

    batch = client.batches.create(
        input_file_id=uploaded.id,
        endpoint="/v1/chat/completions",
        completion_window="24h"
    )

    batch_ids.append(batch.id)
    print(f"  Submitted batch {idx+1}/{len(chunks)} — {len(chunk)} rows — ID: {batch.id}")

# ── Save all batch IDs ───────────────────────────────────────────
with open("data/recover_batch_ids.txt", "w") as f:
    for bid in batch_ids:
        f.write(bid + "\n")

print(f"\nAll {len(batch_ids)} batches submitted.")
print("Saved batch IDs to data/recover_batch_ids.txt")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# SCRIPT 3b: recover_incorrect_batch_collect.py
# Polls ALL batch jobs and appends results to data/recovered.jsonl
# No answer checking — just save everything and let the judger score
# ═══════════════════════════════════════════════════════════════════
import json, os, time
from openai import OpenAI

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# ── Load all batch IDs ───────────────────────────────────────────
with open("data/recover_batch_ids.txt", "r") as f:
    batch_ids = [line.strip() for line in f if line.strip()]
print(f"Tracking {len(batch_ids)} batches")

# ── Poll until ALL batches complete ─────────────────────────────
while True:
    statuses = []
    for batch_id in batch_ids:
        batch  = client.batches.retrieve(batch_id)
        counts = batch.request_counts
        print(f"  {batch_id} | {batch.status} | {counts.completed}/{counts.total} | failed: {counts.failed}")
        statuses.append(batch.status)

    done = all(s in ("completed", "failed", "expired", "cancelled") for s in statuses)
    if done:
        print("All batches finished!")
        break
    print("--- waiting 60s ---\n")
    time.sleep(60)

# ── Load original incorrect rows for metadata ────────────────────
with open("data/incorrect.jsonl", "r") as f:
    incorrect_lookup = {str(json.loads(line)["id"]): json.loads(line) for line in f}

# ── Collect and save ─────────────────────────────────────────────
recovered = []

for batch_id in batch_ids:
    batch = client.batches.retrieve(batch_id)

    if batch.status != "completed":
        print(f"  Skipping {batch_id} — status: {batch.status}")
        continue

    content = client.files.content(batch.output_file_id).text
    results = [json.loads(line) for line in content.strip().split("\n") if line.strip()]
    print(f"  {batch_id} — {len(results)} results downloaded")

    for result in results:
        custom_id = result["custom_id"]
        row       = incorrect_lookup.get(custom_id)
        if row is None:
            continue

        try:
            response = result["response"]["body"]["choices"][0]["message"]["content"]
        except (KeyError, IndexError):
            print(f"    Skipping id={custom_id} — missing response")
            continue

        recovered.append({
            **row,
            "reasoning_raw": response
        })

print(f"\nCollected {len(recovered)} rows")

with open("data/recovered.jsonl", "w") as f:
    for row in recovered:
        f.write(json.dumps(row) + "\n")

print("Saved to data/recovered.jsonl")

In [ ]:
import pandas as pd, re, sys, json
from tqdm import tqdm

# ── 1. Convert JSONL to CSV ──────────────────────────────────────────────────
with open("data/recovered.jsonl", "r") as f:
    raw_data = [json.loads(line) for line in f]

# CSV has just question + reasoning_raw (the model's response)
csv_df = pd.DataFrame([{
    "id":           row["id"],
    "question":     row["question"],
    "revised_response": row["reasoning_raw"]   # renamed to match reference script
} for row in raw_data])

csv_df.to_csv("data/recovered_formatted.csv", index=False)
print(f"Converted {len(csv_df)} rows to data/recovered_formatted.csv")

In [ ]:
df   = pd.read_csv("data/recovered_formatted.csv")
data = raw_data  # ground truth lookup

print(f"Loaded {len(df)} responses from checkpoint")

def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", str(text).upper())
    return matches[-1] if matches else ""

def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()

sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

data_lookup = {item["id"]: item for item in data}

results = []
for _, row in tqdm(df.iterrows(), total=len(df), desc="Scoring"):
    item = data_lookup.get(row["id"])
    if item is None:
        continue
    response = row["revised_response"]
    is_mcq   = bool(item.get("options"))
    gold     = item["answer"]

    if is_mcq:
        correct = score_mcq(str(response), str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       row["id"],
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

results_df = pd.DataFrame(results)
total    = len(results_df)
correct  = results_df["correct"].sum()
mcq_acc  = results_df[results_df["is_mcq"]]["correct"].mean()
free_acc = results_df[~results_df["is_mcq"]]["correct"].mean()

print(f"\nScoring complete: {total} results")
print(f"Overall accuracy:    {correct/total:.4f} ({int(correct)}/{total})")
print(f"MCQ accuracy:        {mcq_acc:.4f}")
print(f"Free-form accuracy:  {free_acc:.4f}")

results_df.to_csv("data/recovered_scoring_results.csv", index=False)
print("Saved to data/recovered_scoring_results.csv")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# SCRIPT 2: split_correct_incorrect.py
# Splits scored rows into two JSONLs:
#   data/correct.jsonl   — rows the model got right (for LoRA training)
#   data/incorrect.jsonl — rows the model got wrong (for recovery)
# reasoning_raw is stripped from both since we'll regenerate it
# ═══════════════════════════════════════════════════════════════════
import json, pandas as pd

# Load scoring results from script 1
results_df = pd.read_csv("data/recovered_scoring_results.csv")

# Load original data to get full row content
with open("data/recovered.jsonl", "r") as f:
    raw_data = {json.loads(line)["id"]: json.loads(line) for line in f}

# Re-open since we consumed the iterator above
with open("data/recovered.jsonl", "r") as f:
    raw_data = {}
    for line in f:
        row = json.loads(line)
        raw_data[row["id"]] = row

correct_ids   = set(results_df[results_df["correct"] == True]["id"])
incorrect_ids = set(results_df[results_df["correct"] == False]["id"])

with open("data/recovered_correct.jsonl", "w") as fc, \
     open("data/recovered_incorrect.jsonl", "w") as fi:
    for id_, row in raw_data.items():
        # keep reasoning_raw in correct rows — these are good training examples
        # strip it from incorrect rows — we'll regenerate better reasoning
        if id_ in correct_ids:
            fc.write(json.dumps(row) + "\n")
        elif id_ in incorrect_ids:
            row_copy = {k: v for k, v in row.items() if k != "reasoning_raw"}
            fi.write(json.dumps(row_copy) + "\n")

print(f"Correct rows:   {len(correct_ids)} → data/recovered_correct.jsonl")
print(f"Incorrect rows: {len(incorrect_ids)} → data/recovered_incorrect.jsonl")

In [ ]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Think concisely. Step by step.\n\n"
    "MANDATORY RULES:\n"
    "- Show ALL reasoning inside <think>...</think> tags.\n"
    "- The FINAL token MUST be \\boxed{} containing your answer.\n"
    "- If multiple sub-answers are required, put them in one box separated by commas, e.g. \\boxed{3, 7}.\n"
    "- Do NOT write \\boxed{final_answer} — replace final_answer with your actual answer.\n\n"
    "FORMAT:\n"
    "<think>\n"
    "[your reasoning here]\n"
    "</think>\n"
    "\\boxed{your_actual_answer}\n"
)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# SCRIPT 3a: recover_incorrect_batch_submit.py
# Submits backward reasoning requests as multiple smaller OpenAI
# batch jobs (50% discount). Smaller batches run faster.
# Run this first, then run 3b to collect results.
# ═══════════════════════════════════════════════════════════════════
import json, os
from openai import OpenAI

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

BATCH_SIZE = 50

# ── Load only free-form incorrect rows ──────────────────────────
with open("data/recovered_incorrect.jsonl", "r") as f:
    incorrect = [json.loads(line) for line in f]

free_form_incorrect = [r for r in incorrect if not r.get("options")]
print(f"Free-form incorrect rows: {len(free_form_incorrect)}")

# ── Split into chunks ────────────────────────────────────────────
chunks = [free_form_incorrect[i:i+BATCH_SIZE]
          for i in range(0, len(free_form_incorrect), BATCH_SIZE)]
print(f"Splitting into {len(chunks)} batches of ~{BATCH_SIZE}")

# ── Submit each chunk as a separate batch ────────────────────────
batch_ids = []

for idx, chunk in enumerate(chunks):
    batch_requests = []
    for row in chunk:
        question = row["question"]
        gold     = str(row["answer"]).strip()

        prompt = (
            f"Solve this math problem carefully.\n\n{question}\n\n"
            f"The correct answer is {gold}. "
            f"Show the complete step-by-step reasoning that arrives at this answer."
        )

        batch_requests.append({
            "custom_id": str(row["id"]),
            "method": "POST",
            "url": "/v1/chat/completions",
            "body": {
                "model": "o4-mini",
                "reasoning_effort": "high",
                "max_completion_tokens": 12000,
                "messages": [
                    {"role": "system", "content": SYSTEM_PROMPT_MATH},
                    {"role": "user",   "content": prompt}
                ]
            }
        })

    # write chunk to temp file
    batch_input_path = f"data/recover_batch_input_{idx}.jsonl"
    with open(batch_input_path, "w") as f:
        for req in batch_requests:
            f.write(json.dumps(req) + "\n")

    # upload and submit
    with open(batch_input_path, "rb") as f:
        uploaded = client.files.create(file=f, purpose="batch")

    batch = client.batches.create(
        input_file_id=uploaded.id,
        endpoint="/v1/chat/completions",
        completion_window="24h"
    )

    batch_ids.append(batch.id)
    print(f"  Submitted batch {idx+1}/{len(chunks)} — {len(chunk)} rows — ID: {batch.id}")

# ── Save all batch IDs ───────────────────────────────────────────
with open("data/recover_batch_ids.txt", "w") as f:
    for bid in batch_ids:
        f.write(bid + "\n")

print(f"\nAll {len(batch_ids)} batches submitted.")
print("Saved batch IDs to data/recover_batch_ids.txt")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# SCRIPT 3b: recover_incorrect_batch_collect.py
# Polls ALL batch jobs and appends results to data/recovered.jsonl
# No answer checking — just save everything and let the judger score
# ═══════════════════════════════════════════════════════════════════
import json, os, time
from openai import OpenAI

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# ── Load all batch IDs ───────────────────────────────────────────
with open("data/recover_batch_ids.txt", "r") as f:
    batch_ids = [line.strip() for line in f if line.strip()]
print(f"Tracking {len(batch_ids)} batches")

# ── Poll until ALL batches complete ─────────────────────────────
while True:
    statuses = []
    for batch_id in batch_ids:
        batch  = client.batches.retrieve(batch_id)
        counts = batch.request_counts
        print(f"  {batch_id} | {batch.status} | {counts.completed}/{counts.total} | failed: {counts.failed}")
        statuses.append(batch.status)

    done = all(s in ("completed", "failed", "expired", "cancelled") for s in statuses)
    if done:
        print("All batches finished!")
        break
    print("--- waiting 60s ---\n")
    time.sleep(60)

# ── Load original incorrect rows for metadata ────────────────────
with open("data/recovered_incorrect.jsonl", "r") as f:
    incorrect_lookup = {str(json.loads(line)["id"]): json.loads(line) for line in f}

# ── Collect and save ─────────────────────────────────────────────
recovered = []

for batch_id in batch_ids:
    batch = client.batches.retrieve(batch_id)

    if batch.status != "completed":
        print(f"  Skipping {batch_id} — status: {batch.status}")
        continue

    content = client.files.content(batch.output_file_id).text
    results = [json.loads(line) for line in content.strip().split("\n") if line.strip()]
    print(f"  {batch_id} — {len(results)} results downloaded")

    for result in results:
        custom_id = result["custom_id"]
        row       = incorrect_lookup.get(custom_id)
        if row is None:
            continue

        try:
            response = result["response"]["body"]["choices"][0]["message"]["content"]
        except (KeyError, IndexError):
            print(f"    Skipping id={custom_id} — missing response")
            continue

        recovered.append({
            **row,
            "reasoning_raw": response
        })

print(f"\nCollected {len(recovered)} rows")

with open("data/recovered2.jsonl", "w") as f:
    for row in recovered:
        f.write(json.dumps(row) + "\n")

print("Saved to data/recovered2.jsonl")

In [ ]:
import pandas as pd, re, sys, json
from tqdm import tqdm

# ── 1. Convert JSONL to CSV ──────────────────────────────────────────────────
with open("data/recovered2.jsonl", "r") as f:
    raw_data = [json.loads(line) for line in f]

# CSV has just question + reasoning_raw (the model's response)
csv_df = pd.DataFrame([{
    "id":           row["id"],
    "question":     row["question"],
    "revised_response": row["reasoning_raw"]   # renamed to match reference script
} for row in raw_data])

csv_df.to_csv("data/recovered_formatted2.csv", index=False)
print(f"Converted {len(csv_df)} rows to data/recovered_formatted2.csv")

In [ ]:
df   = pd.read_csv("data/recovered_formatted2.csv")
data = raw_data  # ground truth lookup

print(f"Loaded {len(df)} responses from checkpoint")

def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", str(text).upper())
    return matches[-1] if matches else ""

def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()

sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

data_lookup = {item["id"]: item for item in data}

results = []
for _, row in tqdm(df.iterrows(), total=len(df), desc="Scoring"):
    item = data_lookup.get(row["id"])
    if item is None:
        continue
    response = row["revised_response"]
    is_mcq   = bool(item.get("options"))
    gold     = item["answer"]

    if is_mcq:
        correct = score_mcq(str(response), str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       row["id"],
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

results_df = pd.DataFrame(results)
total    = len(results_df)
correct  = results_df["correct"].sum()
mcq_acc  = results_df[results_df["is_mcq"]]["correct"].mean()
free_acc = results_df[~results_df["is_mcq"]]["correct"].mean()

print(f"\nScoring complete: {total} results")
print(f"Overall accuracy:    {correct/total:.4f} ({int(correct)}/{total})")
print(f"MCQ accuracy:        {mcq_acc:.4f}")
print(f"Free-form accuracy:  {free_acc:.4f}")

results_df.to_csv("data/recovered_scoring_results2.csv", index=False)
print("Saved to data/recovered_scoring_results2.csv")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# SCRIPT 2: split_correct_incorrect.py
# Splits scored rows into two JSONLs:
#   data/correct.jsonl   — rows the model got right (for LoRA training)
#   data/incorrect.jsonl — rows the model got wrong (for recovery)
# reasoning_raw is stripped from both since we'll regenerate it
# ═══════════════════════════════════════════════════════════════════
import json, pandas as pd

# Load scoring results from script 1
results_df = pd.read_csv("data/recovered_scoring_results2.csv")

# Load original data to get full row content
with open("data/recovered2.jsonl", "r") as f:
    raw_data = {json.loads(line)["id"]: json.loads(line) for line in f}

# Re-open since we consumed the iterator above
with open("data/recovered2.jsonl", "r") as f:
    raw_data = {}
    for line in f:
        row = json.loads(line)
        raw_data[row["id"]] = row

correct_ids   = set(results_df[results_df["correct"] == True]["id"])
incorrect_ids = set(results_df[results_df["correct"] == False]["id"])

with open("data/recovered_correct2.jsonl", "w") as fc, \
     open("data/recovered_incorrect2.jsonl", "w") as fi:
    for id_, row in raw_data.items():
        # keep reasoning_raw in correct rows — these are good training examples
        # strip it from incorrect rows — we'll regenerate better reasoning
        if id_ in correct_ids:
            fc.write(json.dumps(row) + "\n")
        elif id_ in incorrect_ids:
            row_copy = {k: v for k, v in row.items() if k != "reasoning_raw"}
            fi.write(json.dumps(row_copy) + "\n")

print(f"Correct rows:   {len(correct_ids)} → data/recovered_correct2.jsonl")
print(f"Incorrect rows: {len(incorrect_ids)} → data/recovered_incorrect2.jsonl")

In [ ]:
#condensing all corrct ines so far

# ═══════════════════════════════════════════════════════════════════
# merge_correct.py
# Merges all correct JSONLs into one, deduplicating by id.
# Later files take priority if same id appears multiple times
# (so recovered versions overwrite original where they overlap)
# ═══════════════════════════════════════════════════════════════════
import json

files = [
    "data/correct.jsonl",
    "data/recovered_correct.jsonl",
    "data/recovered_correct2.jsonl",
]

merged = {}  # id -> row, later files overwrite earlier ones

for path in files:
    try:
        with open(path, "r") as f:
            rows = [json.loads(line) for line in f if line.strip()]
        print(f"{path}: {len(rows)} rows")
        for row in rows:
            merged[row["id"]] = row
    except FileNotFoundError:
        print(f"  Skipping {path} — not found")

print(f"\nTotal after dedup: {len(merged)} rows")

with open("data/all_correct.jsonl", "w") as f:
    for row in merged.values():
        f.write(json.dumps(row) + "\n")

print("Saved to data/all_correct.jsonl")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# SCRIPT 3a: recover_incorrect_batch_submit.py
# Submits backward reasoning requests as multiple smaller OpenAI
# batch jobs (50% discount). Smaller batches run faster.
# Run this first, then run 3b to collect results.
# ═══════════════════════════════════════════════════════════════════
import json, os
from openai import OpenAI

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

BATCH_SIZE = 50

# ── Load only free-form incorrect rows ──────────────────────────
with open("data/recovered_incorrect2.jsonl", "r") as f:
    incorrect = [json.loads(line) for line in f]

free_form_incorrect = [r for r in incorrect if not r.get("options")]
print(f"Free-form incorrect rows: {len(free_form_incorrect)}")

# ── Split into chunks ────────────────────────────────────────────
chunks = [free_form_incorrect[i:i+BATCH_SIZE]
          for i in range(0, len(free_form_incorrect), BATCH_SIZE)]
print(f"Splitting into {len(chunks)} batches of ~{BATCH_SIZE}")

# ── Submit each chunk as a separate batch ────────────────────────
batch_ids = []

for idx, chunk in enumerate(chunks):
    batch_requests = []
    for row in chunk:
        question = row["question"]
        gold     = str(row["answer"]).strip()

        prompt = (
            f"Solve this math problem carefully.\n\n{question}\n\n"
            f"The correct answer is {gold}. "
            f"Show the complete step-by-step reasoning that arrives at this answer."
        )

        batch_requests.append({
            "custom_id": str(row["id"]),
            "method": "POST",
            "url": "/v1/chat/completions",
            "body": {
                "model": "o4-mini",
                "reasoning_effort": "high",
                "max_completion_tokens": 12000,
                "messages": [
                    {"role": "system", "content": SYSTEM_PROMPT_MATH},
                    {"role": "user",   "content": prompt}
                ]
            }
        })

    # write chunk to temp file
    batch_input_path = f"data/recover_batch_input_{idx}.jsonl"
    with open(batch_input_path, "w") as f:
        for req in batch_requests:
            f.write(json.dumps(req) + "\n")

    # upload and submit
    with open(batch_input_path, "rb") as f:
        uploaded = client.files.create(file=f, purpose="batch")

    batch = client.batches.create(
        input_file_id=uploaded.id,
        endpoint="/v1/chat/completions",
        completion_window="24h"
    )

    batch_ids.append(batch.id)
    print(f"  Submitted batch {idx+1}/{len(chunks)} — {len(chunk)} rows — ID: {batch.id}")

# ── Save all batch IDs ───────────────────────────────────────────
with open("data/recover_batch_ids.txt", "w") as f:
    for bid in batch_ids:
        f.write(bid + "\n")

print(f"\nAll {len(batch_ids)} batches submitted.")
print("Saved batch IDs to data/recover_batch_ids.txt")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# SCRIPT 3b: recover_incorrect_batch_collect.py
# Polls ALL batch jobs and appends results to data/recovered.jsonl
# No answer checking — just save everything and let the judger score
# ═══════════════════════════════════════════════════════════════════
import json, os, time
from openai import OpenAI

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# ── Load all batch IDs ───────────────────────────────────────────
with open("data/recover_batch_ids.txt", "r") as f:
    batch_ids = [line.strip() for line in f if line.strip()]
print(f"Tracking {len(batch_ids)} batches")

# ── Poll until ALL batches complete ─────────────────────────────
while True:
    statuses = []
    for batch_id in batch_ids:
        batch  = client.batches.retrieve(batch_id)
        counts = batch.request_counts
        print(f"  {batch_id} | {batch.status} | {counts.completed}/{counts.total} | failed: {counts.failed}")
        statuses.append(batch.status)

    done = all(s in ("completed", "failed", "expired", "cancelled") for s in statuses)
    if done:
        print("All batches finished!")
        break
    print("--- waiting 60s ---\n")
    time.sleep(60)

# ── Load original incorrect rows for metadata ────────────────────
with open("data/recovered_incorrect2.jsonl", "r") as f:
    incorrect_lookup = {str(json.loads(line)["id"]): json.loads(line) for line in f}

# ── Collect and save ─────────────────────────────────────────────
recovered = []

for batch_id in batch_ids:
    batch = client.batches.retrieve(batch_id)

    if batch.status != "completed":
        print(f"  Skipping {batch_id} — status: {batch.status}")
        continue

    content = client.files.content(batch.output_file_id).text
    results = [json.loads(line) for line in content.strip().split("\n") if line.strip()]
    print(f"  {batch_id} — {len(results)} results downloaded")

    for result in results:
        custom_id = result["custom_id"]
        row       = incorrect_lookup.get(custom_id)
        if row is None:
            continue

        try:
            response = result["response"]["body"]["choices"][0]["message"]["content"]
        except (KeyError, IndexError):
            print(f"    Skipping id={custom_id} — missing response")
            continue

        recovered.append({
            **row,
            "reasoning_raw": response
        })

print(f"\nCollected {len(recovered)} rows")

with open("data/recovered3.jsonl", "w") as f:
    for row in recovered:
        f.write(json.dumps(row) + "\n")

print("Saved to data/recovered3.jsonl")

In [ ]:
import pandas as pd, re, sys, json
from tqdm import tqdm

# ── 1. Convert JSONL to CSV ──────────────────────────────────────────────────
with open("data/recovered3.jsonl", "r") as f:
    raw_data = [json.loads(line) for line in f]

# CSV has just question + reasoning_raw (the model's response)
csv_df = pd.DataFrame([{
    "id":           row["id"],
    "question":     row["question"],
    "revised_response": row["reasoning_raw"]   # renamed to match reference script
} for row in raw_data])

csv_df.to_csv("data/recovered_formatted3.csv", index=False)
print(f"Converted {len(csv_df)} rows to data/recovered_formatted3.csv")

In [ ]:
df   = pd.read_csv("data/recovered_formatted3.csv")
data = raw_data  # ground truth lookup

print(f"Loaded {len(df)} responses from checkpoint")

def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", str(text).upper())
    return matches[-1] if matches else ""

def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()

sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

data_lookup = {item["id"]: item for item in data}

results = []
for _, row in tqdm(df.iterrows(), total=len(df), desc="Scoring"):
    item = data_lookup.get(row["id"])
    if item is None:
        continue
    response = row["revised_response"]
    is_mcq   = bool(item.get("options"))
    gold     = item["answer"]

    if is_mcq:
        correct = score_mcq(str(response), str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       row["id"],
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

results_df = pd.DataFrame(results)
total    = len(results_df)
correct  = results_df["correct"].sum()
mcq_acc  = results_df[results_df["is_mcq"]]["correct"].mean()
free_acc = results_df[~results_df["is_mcq"]]["correct"].mean()

print(f"\nScoring complete: {total} results")
print(f"Overall accuracy:    {correct/total:.4f} ({int(correct)}/{total})")
print(f"MCQ accuracy:        {mcq_acc:.4f}")
print(f"Free-form accuracy:  {free_acc:.4f}")

results_df.to_csv("data/recovered_scoring_results3.csv", index=False)
print("Saved to data/recovered_scoring_results3.csv")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# SCRIPT 2: split_correct_incorrect.py
# Splits scored rows into two JSONLs:
#   data/correct.jsonl   — rows the model got right (for LoRA training)
#   data/incorrect.jsonl — rows the model got wrong (for recovery)
# reasoning_raw is stripped from both since we'll regenerate it
# ═══════════════════════════════════════════════════════════════════
import json, pandas as pd

# Load scoring results from script 1
results_df = pd.read_csv("data/recovered_scoring_results3.csv")

# Load original data to get full row content
with open("data/recovered3.jsonl", "r") as f:
    raw_data = {json.loads(line)["id"]: json.loads(line) for line in f}

# Re-open since we consumed the iterator above
with open("data/recovered2.jsonl", "r") as f:
    raw_data = {}
    for line in f:
        row = json.loads(line)
        raw_data[row["id"]] = row

correct_ids   = set(results_df[results_df["correct"] == True]["id"])
incorrect_ids = set(results_df[results_df["correct"] == False]["id"])

with open("data/recovered_correct3.jsonl", "w") as fc, \
     open("data/recovered_incorrect3.jsonl", "w") as fi:
    for id_, row in raw_data.items():
        # keep reasoning_raw in correct rows — these are good training examples
        # strip it from incorrect rows — we'll regenerate better reasoning
        if id_ in correct_ids:
            fc.write(json.dumps(row) + "\n")
        elif id_ in incorrect_ids:
            row_copy = {k: v for k, v in row.items() if k != "reasoning_raw"}
            fi.write(json.dumps(row_copy) + "\n")

print(f"Correct rows:   {len(correct_ids)} → data/recovered_correct3.jsonl")
print(f"Incorrect rows: {len(incorrect_ids)} → data/recovered_incorrect3.jsonl")

In [ ]:
#condensing all corrct ines so far

# ═══════════════════════════════════════════════════════════════════
# merge_correct.py
# Merges all correct JSONLs into one, deduplicating by id.
# Later files take priority if same id appears multiple times
# (so recovered versions overwrite original where they overlap)
# ═══════════════════════════════════════════════════════════════════
import json

files = [
    "data/all_correct.jsonl",
    "data/recovered_correct3.jsonl",
]

merged = {}  # id -> row, later files overwrite earlier ones

for path in files:
    try:
        with open(path, "r") as f:
            rows = [json.loads(line) for line in f if line.strip()]
        print(f"{path}: {len(rows)} rows")
        for row in rows:
            merged[row["id"]] = row
    except FileNotFoundError:
        print(f"  Skipping {path} — not found")

print(f"\nTotal after dedup: {len(merged)} rows")

with open("data/final_correct.jsonl", "w") as f:
    for row in merged.values():
        f.write(json.dumps(row) + "\n")

print("Saved to data/final_correct.jsonl")